# MLP Single Digit Addition

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

## Creates the Dataset

In [2]:
X       = torch.zeros(100, 20)
Y_ones  = torch.zeros(100, dtype=torch.long)
Y_carry = torch.zeros(100, dtype=torch.long)

for a in range(10):
    for b in range(10):
        i = a * 10 + b
        X[i, a]      = 1.0
        X[i, 10 + b] = 1.0
        Y_ones[i]    = (a + b) % 10
        Y_carry[i]   = (a + b) // 10

print('Dataset created')

Dataset created


## Architecture

input (20) -> hidden layer 1 (64) (frozen in Model C) -> hidden layer 2 (64) -> ones head (10 classes) and carry head (2 classes)


In [3]:
class DigitMLP(nn.Module):
    def __init__(self, freeze_layer1=False):
        super().__init__()
        self.hidden1    = nn.Linear(20, 64)
        self.hidden2    = nn.Linear(64, 64)
        self.ones_head  = nn.Linear(64, 10)
        self.carry_head = nn.Linear(64,  2)

        #locks hidden1 for Model C
        if freeze_layer1: 
            for p in self.hidden1.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        h1 = self.hidden1(x).clamp(min=0)   
        h2 = self.hidden2(h1).clamp(min=0)
        return self.ones_head(h2), self.carry_head(h2)

def accuracy(logits, targets):
    return (logits.argmax(1) == targets).sum().item()

print('Architecture defined')

Architecture defined


## Helper
trains the model and takes the heads you want to be part of the training as input parameters

In [4]:
def train(model, heads='both', max_epochs=50000, lr=1e-3):

    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.Adam(params, lr=lr)
    ce     = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        opt.zero_grad()
        lo, lc = model(X)
        loss = ce(lo, Y_ones)
        if heads == 'both':
            loss = loss + ce(lc, Y_carry)
        loss.backward()
        opt.step()

        co = accuracy(lo, Y_ones)
        cc = accuracy(lc, Y_carry)
        if epoch % 5000 == 0:
            print(f'  epoch {epoch:5d}  loss={loss.item():.4f}  ones={co}/100  carry={cc}/100')
        done = (co == 100) if heads == 'ones' else (co == 100 and cc == 100)
        if done:
            print(f'  100% at epoch {epoch}')
            break
    return time.time() - t0

print('Training helper defined')

Training helper defined


## Model A — ones digit only

In [5]:
model_a = DigitMLP()
print('Training Model A (ones only)...')
t_a = train(model_a, heads='ones')
print(f'Model A training time: {t_a:.2f}s')

model_a.eval()
for p in model_a.parameters():
    p.requires_grad_(False)

Training Model A (ones only)...
  100% at epoch 169
Model A training time: 0.25s


## Verify Model A

In [6]:
with torch.no_grad():
    lo, lc = model_a(X)
    print(f'Model A  ones accuracy : {accuracy(lo, Y_ones)}/100')
    print(f'Model A  carry accuracy: {accuracy(lc, Y_carry)}/100  (untrained)')

Model A  ones accuracy : 100/100
Model A  carry accuracy: 44/100  (untrained)


## Model B — both heads from scratch

In [7]:
model_b = DigitMLP()
print('Training Model B (both heads from scratch)')
t_b = train(model_b, heads='both')
print(f'Model B training time: {t_b:.2f}s')

model_b.eval()
for p in model_b.parameters():
    p.requires_grad_(False)

Training Model B (both heads from scratch)
  100% at epoch 252
Model B training time: 0.41s


## Verify Model B

In [8]:
with torch.no_grad():
    lo, lc = model_b(X)
    print(f'Model B  ones accuracy : {accuracy(lo, Y_ones)}/100')
    print(f'Model B  carry accuracy: {accuracy(lc, Y_carry)}/100')

Model B  ones accuracy : 100/100
Model B  carry accuracy: 100/100


## Model C — fine-tune from Model A with first layer frozen

load_state_dict copies all of Model A's weights. Only hidden2, ones_head, and carry_head receive gradients.

In [9]:
model_c = DigitMLP(freeze_layer1=True)
model_c.load_state_dict(model_a.state_dict())   # warm-start from Model A

for name, p in model_c.named_parameters():
    if 'hidden1' not in name:
        p.requires_grad_(True)

print('Model C: hidden1 frozen, all other layers trainable')
t_c = train(model_c, heads='both')
print(f'Model C training time: {t_c:.2f}s')
model_c.eval()

Model C: hidden1 frozen, all other layers trainable
  100% at epoch 125
Model C training time: 0.16s


DigitMLP(
  (hidden1): Linear(in_features=20, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=64, bias=True)
  (ones_head): Linear(in_features=64, out_features=10, bias=True)
  (carry_head): Linear(in_features=64, out_features=2, bias=True)
)

## Verify Model C

In [10]:
with torch.no_grad():
    lo, lc = model_c(X)
    print(f'Model C  ones accuracy : {accuracy(lo, Y_ones)}/100')
    print(f'Model C  carry accuracy: {accuracy(lc, Y_carry)}/100')

Model C  ones accuracy : 100/100
Model C  carry accuracy: 100/100


## Compare training times

In [11]:
print('=' * 40)
print(f'Model A (ones only, from scratch) : {t_a:.2f}s')
print(f'Model B (both heads, from scratch) : {t_b:.2f}s')
print(f'Model C (both heads, frozen h1)    : {t_c:.2f}s')
print('=' * 40)
if t_c < t_b:
    print(f'C was faster than B by {t_b - t_c:.2f}s')
else:
    print(f'B was faster than C by {t_c - t_b:.2f}s')
if t_c + t_a < t_b:
    print(f'A+C together were faster than B by {t_b - (t_c + t_a):.2f}s')
else:
    print(f'B was faster than A+C together by {t_c + t_a - t_b:.2f}s')

Model A (ones only, from scratch) : 0.25s
Model B (both heads, from scratch) : 0.41s
Model C (both heads, frozen h1)    : 0.16s
C was faster than B by 0.24s
B was faster than A+C together by 0.01s
